In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.memory import ChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# 일반 언어 모델과 채팅 모델

# 채팅의 가장 큰 특징은 이전 대화의 맥락을 기억하고 활용하는 것
# 이전 대화의 맥락을 기억하고 활용하는 방법은 크게 두 가지임

# 일반 언어 모델의 경우 프롬프트는 단일 문자열이고 응답도 단일 문자열
# 채팅 언어 모델의 경우 프롬프트는 list of messages

# 1. ChatPromptTemplate를 사용하여 시스템 메시지, 사용자 메시지, AI 메시지 등을 순서대로 조합하여 하나의 완성된 프롬프트를 만드는 방법
# 2. 메모리 기능 활용 방식

# 즉, 채팅 언어 모델의 프롬프트는 시스템 메시지, 사용자 메시지, AI 메시지 등을 순서대로 조합해서 대화 맥락을 만듬
# 채팅 모델은 프롬프트내에 있는 list of messages를 보고 답변을 생성

# 메시지에는(모두 BaseMessage의 자식 클래스) 
# SystemMessage: 대화의 전체적인 맥락, 규칙, 또는 AI의 페르소나를 정의
# HumanMessage: 사용자가 입력하는 질문이나 메시지
# AIMessage: AI의 응답

# ToolMessage: AI가 외부 도구를 사용해서 얻은 결과
# AIMessageChunk: 답변을 조각으로 나누어 보내는(streaming) 경우의 "응답 한 조각"

# 채팅 모델 프롬프트 예제: 다른 말로 'messages'
# [
#     SystemMessage(content="당신은 한국어로 질문에 답변하는 친절한 AI 비서입니다."),
#     ("system", "당신은 한국어로 질문에 답변하는 친절한 AI 비서입니다.")

#     HumanMessage(content="서울의 날씨에 대해 알려주세요."),
#     ("human", "서울의 날씨에 대해 알려주세요.")

#     AIMessage(content="서울의 현재 날씨는 맑고 기온은 25도입니다."),
#     ("assistant", "서울의 현재 날씨는 맑고 기온은 25도입니다.")

#     HumanMessage(content="내일 날씨도 알려줄 수 있나요?")
# ]

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage(content="당신은 사용자의 질문에 짧게 답변하는 AI입니다.")
human_msg = HumanMessage(content="안녕?")
ai_msg = AIMessage(content="안녕하세요! 궁금한 점이 있으신가요?")

print(system_msg)
print(human_msg)
print(ai_msg)

example = HumanMessage(content="내 이름은 {name}이야.")
# content로 넘긴 "내 이름은 {name}이야."은 완성된 메시지로서
# {name}을 변수로 인식하지 않음
print(example)

In [ ]:

# 일반 언어 모델에서 사용하는 템플릿에는 PromptTemplate, FewShotPromptTemplate
# 채팅 언어 모델에서 사용하는 템플릿에는 ChatPromptTemplate, FewShotChatMessagePromptTemplate

In [ ]:
# ChatPromptTemplate

# 채팅 언어 모델과의 대화를 구성하기 위해 사용되는 프롬프트 템플릿
# 일반 텍스트가 아닌, 역할을 가진 메시지들의 리스트가 내용

# ChatPromptTemplate을 사용하는 방법

# 1. from_messages()
# 프롬프트 템플릿을 정의하고 초기화
# 두 가지 형태의 인자 (둘 다 변수를 포함시킬 수 있음)
# ⓐ list of tuples: [("system", "내용"), ("human", "내용")]
# 
# ⓑ list of BaseMessage: [SystemMessage(content="내용"), HumanMessage(content="내용")]
# 

# ChatPromptTemplate.from_messages([
#           ("system", "당신은 유용한 챗봇입니다."), 
#           ("human", "안녕하세요, 제 이름은 {name}입니다.")
# ])


# 2. format_messages()
# 템플릿에 값을 채워 최종 메시지 리스트를 생성

# template.format_messages(name="정호")

# invoke()는 값을 채우면서 실행

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다."),
    ("human", "{input}")
])

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

chain = prompt | chat

response_1 = chain.invoke({"role": "친절한 여행 가이드", "input": "제주도 가볼만한 곳을 알려주세요."})
print(f"--- 첫 번째 응답 (친절한 여행 가이드) ---")
print(response_1.content)

print("\n" + "="*50 + "\n")

response_2 = chain.invoke({"role": "프랑스 요리 전문가", "input": "마카롱 만드는 법을 알려주세요."})
print(f"--- 두 번째 응답 (프랑스 요리 전문가) ---")
print(response_2.content)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

template = ChatPromptTemplate.from_messages([
    ("human", "내 이름은 {name}이야."),
    ("assistant", "안녕하세요, {name}!")
])

print(type(template))  # <class 'langchain_core.prompts.chat.ChatPromptTemplate'>

final_messages = template.format_messages(name="제이크")
# final_messages는 HumanMessage, AIMessage의 리스트

print('-'*50)
print(type(final_messages))  # <class 'list'>
for x in final_messages:
    print(type(x), x.content)

In [ ]:
# invoke() vs. format_messages()
# 공통점은 값을 채운다는 점

# invoke()는 다음 Runnable로 전달할 수 있는 형태로 반환
# format_messages()의 리턴은 Runnable이 아님 → 단순 list of messages를 리턴

# from_messages는 ChatPromptTemplate 객체를 만드는 데 사용
# format_messages는 만들어진 ChatPromptTemplate에 값을 채워 최종 메시지 리스트를 만드는 데 사용

from langchain.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

messages = [
    HumanMessage(content="내 이름은 {name}이야.")
    # content로 넘긴 "내 이름은 {name}이야."은 완성된 메시지로서
    # {name}을 변수로 인식하지 않음
]

# from_messages()는 list of (role, content)를 인자로 받음
template = ChatPromptTemplate.from_messages(messages)

formatted_messages = template.format_messages(name="제이크")
print(formatted_messages)

In [ ]:
# HumanMessage() 내부에 변수를 넣으려면 content 인자에 F-문자열을 사용해야 함
name = "정호"
age = 30

message = HumanMessage(content=f"안녕하세요, 제 이름은 {name}이고 나이는 {age}살입니다.")
# message = HumanMessage(content="안녕하세요, 제 이름은 {name}이고 나이는 {age}살입니다.")
print(message)

In [ ]:
# 1. 사용자 → 모델: "안녕하세요, 제주도 여행 계획 중인데, 가볼 만한 곳 추천해 주실 수 있나요?"
# 2. 모델 → 사용자: "안녕하세요! 제주도 여행을 계획 중이시군요. 제주도는 아름다운 곳이 정말 많죠! 오름, 해변, 폭포 등 다양한 명소가 있습니다. 혹시 특별히 관심 있는 여행 테마가 있으신가요?"
# 3. 사용자 → 모델: "네, 오름에 관심이 많아요. 오름을 중심으로 여행할 만한 코스를 추천해 주세요."

# ChatPromptTemplate([SystemMessage(content="당신은 제주도 지역을 전문으로 하는 친절한 여행 가이드입니다.")
#     ,
#     HumanMessage(content="안녕하세요, 제주도 여행 계획 중인데, 가볼 만한 곳 추천해 주실 수 있나요?"),
#     AIMessage(content="안녕하세요! 제주도 여행을 계획 중이시군요. 제주도는 아름다운 곳이 정말 많죠! 오름, 해변, 폭포 등 다양한 명소가 있습니다. 혹시 특별히 관심 있는 여행 테마가 있으신가요?"),
#     HumanMessage(content="네, 오름에 관심이 많아요. 오름을 중심으로 여행할 만한 코스를 추천해 주세요.")
# ])

# SystemMessage는 실제 대화 내용이 아님
# 대화가 진행될수록 이전 대화 내용이 순차적으로 추가되는 방식

In [ ]:
# 채팅 모델에서 invoke의 사용법

# 1. ChatModel을 직접 호출하는 경우 BaseMessage 객체 리스트를 전달해야 함

from langchain_core.messages import SystemMessage, HumanMessage

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# invoke의 인자는 반드시 메시지 리스트
messages = [
    SystemMessage(content="당신은 사용자의 질문에 답하는 AI입니다."),
    HumanMessage(content="내 이름은 제이크야.")
]

response = chat.invoke(messages)
print(response.content)

In [ ]:
# 채팅 모델에서 invoke의 사용법

# 2. Runnable 체인을 호출: 프롬프트 템플릿의 변수들을 담은 딕셔너리를 전달

from langchain.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    # 튜플의 경우 F-문자열 사용하지 않아도 됨
    ("system", "당신은 {role}입니다."),
    ("human", "{input}")
])

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
chain = chat_prompt | chat
response = chain.invoke({"role": "친절한 여행 가이드", "input": "제주도 가볼만한 곳을 알려주세요."})
print(response.content)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

chat = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

print("--- 첫 번째 턴: 대화 시작 ---")

prompt_turn1 = [
    SystemMessage(content="당신은 사용자의 질문에 답하는 AI입니다."),
    HumanMessage(content="나는 파란색을 좋아해. 내 이름은 제이크야.")
]

# 첫 번째 응답
# ChatModel을 직접 호출
# ChatModel을 직접 호출하는 경우 BaseMessage 객체 리스트를 전달해야 함
response_turn1 = chat.invoke(prompt_turn1)
print(f"모델 응답 (Turn 1): {response_turn1.content}")

print("\n--- 두 번째 턴: 이전 대화 맥락 포함 ---")
# 첫 번째 턴의 모델의 응답을 AIMessage로 저장
ai_message_from_turn1 = AIMessage(content=response_turn1.content)

# 모델이 대화의 맥락(context)을 기억하고 일관성 있는 응답을 생성하도록 하기 위해서 이전 대화 내용을 재사용
prompt_turn2 = [
    prompt_turn1[0],  # SystemMessage(content="당신은 사용자의 질문에 답하는 AI입니다.")
    prompt_turn1[1],  # HumanMessage(content="나는 파란색을 좋아해. 내 이름은 제이크야.")
    ai_message_from_turn1, # 모델의 이전 응답 (AIMessage)
    HumanMessage(content="내 이름은 뭐였지?") # 새로운 질문
]

# 두 번째 응답
response_turn2 = chat.invoke(prompt_turn2)
print(f"모델 응답 (Turn 2): {response_turn2.content}")

In [ ]:
# MessagesPlaceholder

# 대화 기록(chat history)이나 동적인 메시지들을 담을 빈 공간을 정의하는 역할
# "여기에 넣어주세요"의 여기가 어디인지 알리는 역할

from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

message_history = [
    HumanMessage(content="내 이름은 제이크야."),
    AIMessage(content="안녕하세요, 제이크! 무엇을 도와드릴까요?")
]

template = ChatPromptTemplate.from_messages([
    SystemMessage(content="당신은 사용자의 질문에 답하는 AI입니다."),
    MessagesPlaceholder(variable_name="history1"), # {history1}로 생각
    HumanMessage(content="내 이름이 뭐였지?")
])

# 프롬프트에 메시지 기록을 채움
# history1이라는 변수에 message_history를 할당하라는 의미
final_prompt = template.format_messages(history1=message_history)

print("--- 완성된 메시지 리스트 ---")
for x in final_prompt:
    print(x)


In [ ]:
# ChatMessageHistory
# 채팅 대화 기록을 관리하고 저장하는 데 사용되는 클래스
# 메모리 기능에 해당

# add_user_message(message: str): 사용자의 메시지를 HumanMessage 객체로 변환하여 추가
# add_ai_message(message: str): AI의 응답 메시지를 AIMessage 객체로 변환하여 추가
# add_message(message: BaseMessage): HumanMessage, AIMessage 등 BaseMessage 객체 유형을 직접 받아 추가

In [ ]:
# ChatMessageHistory - 1

from langchain.memory import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

history = ChatMessageHistory() # list of messages

history.add_user_message("안녕하세요, 랭체인이 뭔가요?")
history.add_ai_message("랭체인은 LLM 애플리케이션 개발을 돕는 프레임워크입니다.")

print(history.messages) # list of messages

In [ ]:
# ChatMessageHistory - 2

from langchain.memory import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage(content="당신은 친절한 AI 비서입니다.")
user_msg = HumanMessage(content="안녕하세요.")
ai_msg = AIMessage(content="안녕하세요! 무엇을 도와드릴까요?")

history = ChatMessageHistory() # [messages]

# add_message()로 메시지 추가
# SystemMessage, HumanMessage, AIMessage 등 BaseMessage의 자식 클래스를 모두 사용 가능
history.add_message(system_msg)
history.add_message(user_msg)
history.add_message(ai_msg)

print(history.messages)

In [ ]:
# RunnableWithMessageHistory

# 이전 대화 기록을 저장하는 용도
# 사용자-모델 간의 대화 내용을 세션 ID(session ID)를 기반으로 저장
# 세션 시작: 새로운 대화 시작, 새로운 세션 ID 부여
# 세션 종료: 앱 종료 등을 통한 대화 종료, 타임아웃

In [ ]:
# session (웹)
# 웹 사이트에 접속하면 서버는 session id를 생성하고 사용자는 브라우저에 쿠키 형태로 저장
# 사용자가 서버에 요청을 할 때마다 session id를 함께 전송
# 서버는 session id를 통해서 저장된 사용자의 상태 정보를 불러옴
# 일정 시간 동안 활동이 없으면 만료되거나 로그아웃 하면 즉시 폐기됨

# session (LangChain)
# 기본적으로 RunnableWithMessageHistory에 전달하는 값은 session_id와 입력에 해당하는 값
# session 정보는 RunnableWithMessageHistory 객체의 invoke() 메서드의 config를 통해서 전달

In [ ]:
# session_id를 저장하는 곳과 메시지 히스토리

# store = {}
# store[session_id] = ChatMessageHistory()
# 세션 아이디는 딕셔너리에, 메시지 히스토리는 ChatMessageHistory() 객체의 형태로 저장

In [ ]:
# RunnableWithMessageHistory

from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.memory import ChatMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI

# 다음 셀에서 계속

In [ ]:

prompt = ChatPromptTemplate.from_messages([
    # 채워야 하는 부분은 history와 input
    # 키 history와 input에 해당하는 실제 값으로 채우게 됨

    # MessagesPlaceholder에 추가할 수 있는 대상은 list of messages [HumanMessage, AIMessage, SystemMessage]
    MessagesPlaceholder(variable_name="history"), # {history}라고 생각하면 됨
    # HumanMessage
    ("human", "{input}")
])

# 다음 셀에서 계속

In [ ]:

chat = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
chain = prompt | chat

# 다음 셀에서 계속

In [ ]:

# 대화 기록을 저장할 간단한 메모리(store) 생성
# 키는 session_id, 값은 해당 세션에 대한 메시지 히스토리
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory() # {'1000': []}
    return store[session_id]

# 다음 셀에서 계속

In [ ]:

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    # 입력은 어디에 있는가?
    input_messages_key="input",
    # get_session_history()로 가져온 과거 메시지에 붙이는 키 값은 무엇?
    history_messages_key="history"

    # {'input': "안녕하세요, 제 이름은 톰입니다.", 'history': ■}가 prompt에 전달
)

# 다음 셀에서 계속

In [ ]:

# 첫 번째 질문
response_1 = with_message_history.invoke(
    {"input": "안녕하세요, 제 이름은 톰입니다."},
    config={"configurable": {"session_id": "user-123"}}
    
)

# with_message_history는 invoke()로 전달받은 {"session_id": "user-123"}로 어떤 대화 기록을 가져올지 결정

# 1.
# session_id인 "user-123"을 get_session_history 함수에 전달
# ChatMessageHistory() 객체를 생성하여 "user-123"를 키 값으로 store = {}에 저장

# 2.
# RunnableWithMessageHistory는 input_messages_key="input"에 의해서 input 값을 현재 대화 메시지로
# +
# RunnableWithMessageHistory는 history_messages_key="history"에 의해서 빈 ChatMessageHistory() 객체를 'history'로 해서
# =
# {'input': '안녕하세요, 제 이름은 톰입니다.', 'history': []}을 만들어서
# 체인에 전달
# 
# 체인은 prompt | chat이고 prompt에는 채워야 할 변수가 'history'와 'input'이 있음
# input에는 현재 대화 메시지가, history에는 세션 아이디에 해당하는 ChatMessageHistory() 객체
# ChatMessageHistory() 객체는 list of BaseMessage로서
# [HumanMessage, AIMessage, SystemMessage]의 형태임

# prompt | chat이 실행이 끝나면 RunnableWithMessageHistory는 사용자의 질문과 답변을 store["user-123"] 즉, 
# ChatMessageHistory() 객체에 추가

print(f"첫 번째 응답: {response_1.content}\n")

# 두 번째 질문 (이전 기록이 포함)
# RunnableWithMessageHistory는 항상 현재 대화 메시지와 해당 세션에 대한 ChatMessageHistory 객체를 결합하여 chain에 전달
response_2 = with_message_history.invoke(
    {"input": "제 이름이 뭐였죠?"},
    config={"configurable": {"session_id": "user-123"}}
    
)
print(f"두 번째 응답: {response_2.content}")

In [ ]:
# RunnableWithMessageHistory - 2

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chat = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 사용자의 질문에 친절하게 답하는 AI 비서입니다. 과거 대화를 참고하여 대답하세요."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

chain = prompt | chat

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    # {'input':, 'history': }
)
# 다음 셀에서 계속

In [ ]:
config1 = {"configurable": {"session_id": "user-123"}}
# 다음 셀에서 계속

In [ ]:
response1 = with_message_history.invoke({"input": "내 이름은 톰이야"}, config=config1)
print(f"-> AI 응답: {response1.content}\n")
# store = {}
# store['user-123'] = [HumanMessage, AIMessage]
# [SystemMessage, HumanMessage]
response2 = with_message_history.invoke({"input": "내 이름이 뭐였지?"}, config=config1)
# {'input': "내 이름이 뭐였지?", 'history': [HumanMessage, AIMessage]}
# [SystemMessage, HumanMessage, AIMessage, HumanMessage]
print(f"-> AI 응답: {response2.content}\n")
# 다음 셀에서 계속

In [ ]:
config2 = {"configurable": {"session_id": "user-456"}}
# 다음 셀에서 계속

In [ ]:
response3 = with_message_history.invoke({"input": "내 이름은 제인이야"}, config=config2)
print(f"-> AI 응답: {response3.content}\n")

response4 = with_message_history.invoke({"input": "내 이름이 뭐였지?"}, config=config2)
print(f"-> AI 응답: {response4.content}\n")
# 다음 셀에서 계속

In [ ]:
print("--- 'store' 저장소의 최종 상태 ---")
print(store.keys())
# 다음 셀에서 계속


In [ ]:
for session_id, history_obj in store.items():
    print(f"세션 ID: '{session_id}'")
    for message in history_obj.messages:
        print(f"  - {message.type.capitalize()}: {message.content}")

In [ ]:
# RunnableWithMessageHistory - 3

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 한국 서울의 날씨를 알려주는 친절한 챗봇입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: ChatMessageHistory(),
    input_messages_key="input",
    history_messages_key="history",
    # ["input": "내일은 어때?", "history": ["오늘 날씨 어때?", "답변"]] >>> prompt (깂을 채우는 용도)
    # [] -> ["오늘 날씨 어때?", "답변"]
)

config = {"configurable": {"session_id": "weather_chat_123"}}

response_1 = chain_with_history.invoke({"input": "오늘 날씨 어때?"}, config=config)
print(f"**사용자**: 오늘 날씨 어때?")
print(f"**AI**: {response_1.content}\n")


response_2 = chain_with_history.invoke({"input": "내일은 어때?"}, config=config)
print(f"**사용자**: 내일은 어때?")
print(f"**AI**: {response_2.content}\n")


In [14]:

print("--- 대화 기록 확인 ---")
# 실제 ChatMessageHistory에 저장된 내용을 직접 확인합니다.
history_obj = ChatMessageHistory()
print(chain_with_history.get_session_history("weather_chat_123").messages)


--- 대화 기록 확인 ---
[]
